In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # for older PyTorch
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # disable gpt.py kernels progress bars
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # for deterministic ops
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
import torch
import pickle

from pathlib import Path
root = str((Path.cwd() / "../..").resolve())
sys.path.insert(0, root) if root not in sys.path else None

from nanorepro.dataloader import DataLoaderSFT
from nanorepro.tasks import TaskMixture, TaskSmolTalk
from nanorepro.tasks import TaskArc, TaskMMLU  # categorical
from nanorepro.tasks import TaskSpellingBee, TaskGSM8K, TaskHumanEval  # generative
from nanorepro.loss_eval import evaluate_bpb
from nanorepro.checkpoint import load_model
from nanorepro.common import get_base_path
from nanorepro.chatcore_eval import evaluate_sft_categorical, evaluate_sft_generative, evaluate_chatcore_metric

BASE_DIR = get_base_path()

In [2]:
class Args:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)
args = Args(
    run="scaling3_6e18_d16-counting",
    no_fa3=True,
    deterministic=True,
    eval_tokens=524288,   # 40*524288,
)

In [3]:
# Compute setup and helpers
device, ddp_master, ddp_rank, ddp_world_size = 'cuda', True, 0, 1
enable_fp8 = False
print0 = print if os.environ.get("RANK", "0") == "0" else lambda *args, **kwargs: None
synchronize = lambda: torch.cuda.synchronize() if device.startswith("cuda") else None
compute_dtype = torch.bfloat16
run_path = os.path.join(BASE_DIR, "runs_sft", args.run if args.run is not None else "default")

In [4]:
# Tokenizer
tok_base_path = os.path.join(BASE_DIR, "tokenizer")
tokenizer_path = os.path.join(tok_base_path, "tokenizer.pkl")
tokenizer = pickle.load(open(tokenizer_path, "rb"))
token_bytes_path = os.path.join(tok_base_path, "token_bytes.pt")
with open(token_bytes_path, "rb") as f:
    token_bytes = torch.load(f, map_location=device)
print0("Vocabulary size:", tokenizer.n_vocab)

Vocabulary size: 32768


In [5]:
# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

# Precision
if device.startswith("cuda"):
    torch.set_float32_matmul_precision("high")  # uses tf32 instead of fp32 for matmuls

# Determinism
# Also need to disable torch.compile for reproducibility
if args.deterministic:
    assert args.no_fa3, "FA3 can't reliably be set to deterministic mode due to bug in upstream implementation"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

In [6]:
# Model Setup
run_name = args.run if args.run is not None else "default"
checkpoints_path = os.path.join(BASE_DIR, "runs_sft", run_name)
model, pretrain_metadata = load_model(
    checkpoints_path=checkpoints_path,
    compute_dtype=compute_dtype,
    enable_fa3=not args.no_fa3,
    fp8_training=enable_fp8,    # doesn't matter
    enable_metrics=False,
    device=device,
    step=None)                  # latest checkpoint
print0("Model configuration:")
for k, v in model.config.to_dict().items():
    print0(f"  {k:>16}: {v}")

Model configuration:
        block_size: 2048
        vocab_size: 32768
           n_layer: 16
            n_head: 8
            n_embd: 1024
    window_pattern: SSSL
        moe_enable: False
       moe_experts: 8
         moe_top_k: 2


In [7]:
# Compile
orig_model = model
if not args.deterministic:
    model = torch.compile(model, dynamic=False)

In [8]:
# Hyperparameter Transfer and Calculation
step = pretrain_metadata["step"]
pretrain_user_cfg = pretrain_metadata["user_config"]
max_seq_len = pretrain_user_cfg['max_seq_len']
micro_batch = pretrain_user_cfg['device_batch_size']
total_batch_size = pretrain_user_cfg["total_batch_size"]
flops_per_token = model.estimate_flops_per_token()

total_flops = step * total_batch_size * flops_per_token

In [9]:
# Eval Dataloader
assert args.eval_tokens % (micro_batch * max_seq_len * ddp_world_size) == 0
eval_steps = args.eval_tokens // (micro_batch * max_seq_len * ddp_world_size)
tasks_eval = TaskMixture([
    TaskSmolTalk(split="test"),                        # 24K tasks
    TaskMMLU(subset="all", split="test", stop=5200),   #  5.2K tasks - match training ratio before repetition (whole test set is 14K)
    TaskGSM8K(subset="main", split="test", stop=420),  #  0.42K tasks (whole test set is 1.32K)
])
eval_loader = DataLoaderSFT(
    tasks=tasks_eval,
    batch_size=micro_batch,
    block_size=max_seq_len,
    tokenizer=tokenizer,
    device=device,
)

In [10]:
# BPB Evaluation
bpb, total_nats, total_bytes = evaluate_bpb(model, token_bytes, eval_loader, eval_steps, device)
print0(f"BPB Eval {step} | BPB {bpb:.14f} | nats {total_nats:.1f} | bytes {total_bytes}")

BPB Eval 969 | BPB 0.31022950461913 | nats 397260.1 | bytes 1847423


In [13]:
task = TaskMMLU("all", "test")
acc, passed, total = evaluate_sft_categorical(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    max_problems=1000
)
print(acc, passed, total)  # 0.338

0.338 338 1000


In [14]:
task = TaskArc("ARC-Easy", "test")
acc, passed, total = evaluate_sft_categorical(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    max_problems=1000
)
print(acc, passed, total)  # 0.478

0.478 478 1000


In [15]:
task = TaskArc("ARC-Challenge", "test")
acc, passed, total = evaluate_sft_categorical(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    max_problems=1000
)
print(acc, passed, total)  # 0.401

0.401 401 1000


In [18]:
task = TaskSpellingBee("test")
acc, passed, total = evaluate_sft_generative(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    num_samples=1,
    temperature=0.0,
    top_k=50,
    max_new_tokens=512,
    max_problems=10
)
print(acc, passed, total)

1.0 10 10


In [19]:
task = TaskGSM8K("main", "test")
acc, passed, total = evaluate_sft_generative(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    num_samples=1,
    temperature=0.0,
    top_k=50,
    max_new_tokens=512,
    max_problems=10
)
print(acc, passed, total)

0.1 1 10


In [20]:
task = TaskHumanEval("test")
acc, passed, total = evaluate_sft_generative(
    task=task,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    num_samples=1,
    temperature=0.0,
    top_k=50,
    max_new_tokens=512,
    max_problems=10
)
print(acc, passed, total)

0.1 1 10


In [25]:
tasks_dict = {
    "arc_easy": TaskArc("ARC-Easy", "test"),
    "arc_challenge": TaskArc("ARC-Challenge", "test"),
    "mmlu": TaskMMLU("all", "test"),
    "gsm8k": TaskGSM8K("main", "test"),
    "human_eval": TaskHumanEval("test"),
    "spelling_bee": TaskSpellingBee("test")
}

In [ ]:
chatcore_metric, chatcore_cat_metric, chatcore_gen_metric, results_list, total_time = evaluate_chatcore_metric(
    tasks_dict=tasks_dict,
    model=model,
    tokenizer=tokenizer,
    micro_batch=micro_batch,
    max_prompt_len=max_seq_len,
    num_samples=1,
    temperature=0.0,
    top_k=50,
    max_new_tokens=512,
    max_problems_cat=None,
    max_problems_gen=24
)

Task         arc_easy (categorical) | df 7.4 | acc 0.4848 | centered_acc 0.4836
Task    arc_challenge (categorical) | df 3.8 | acc 0.3968 | centered_acc 0.3952
Task             mmlu (categorical) | df 75.6 | acc 0.3336 | centered_acc 0.3319
Task            gsm8k (generative 1 samples) | df 82.6 | acc 0.1667 | centered_acc 0.1667
Task       human_eval (generative 1 samples) | df 162.2 | acc 0.0833 | centered_acc 0.0833
Task     spelling_bee (generative 1 samples) | df 87.9 | acc 1.0000 | centered_acc 1.0000


In [ ]:
print(f"ChatCORE metric: {chatcore_metric:.4f}")
print(f"ChatCORE category metric: {chatcore_cat_metric:.4f}")
print(f"ChatCORE generative metric: {chatcore_gen_metric:.4f}")
for r in results_list:
    display(r)
print(f"Total evaluation time: {total_time:.1f} seconds")

ChatCORE metric: 0.2921
ChatCORE category metric: 0.4036


{'task': 'arc_easy',
 'eval_type': 'categorical',
 'accuracy': 0.48484848484848486,
 'passed': 1152,
 'total': 2376,
 'centered_accuracy': 0.4835573782942204}

{'task': 'arc_challenge',
 'eval_type': 'categorical',
 'accuracy': 0.3967576791808874,
 'passed': 465,
 'total': 1172,
 'centered_accuracy': 0.39524579366504997}

{'task': 'mmlu',
 'eval_type': 'categorical',
 'accuracy': 0.3335707164221621,
 'passed': 4684,
 'total': 14042,
 'centered_accuracy': 0.3319004675911399}

{'task': 'gsm8k',
 'eval_type': 'generative',
 'accuracy': 0.16666666666666666,
 'passed': 4,
 'total': 24,
 'centered_accuracy': 0.16666666666666666}

{'task': 'human_eval',
 'eval_type': 'generative',
 'accuracy': 0.08333333333333333,
 'passed': 2,
 'total': 24,
 'centered_accuracy': 0.08333333333333333}

{'task': 'spelling_bee',
 'eval_type': 'generative',
 'accuracy': 1.0,
 'passed': 24,
 'total': 24,
 'centered_accuracy': 1.0}

Total evaluation time: 419.5 seconds
